# requires-grad-propagation — ex2: requires_grad propagation also scans kwargs Tensor values

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `requires-grad-propagation`. Running the final beacon cell reports progress against the `Backprop: requires_grad propagation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: requires_grad propagation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-propagation`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-propagation"
DD_SUBTOPIC = "Backprop: requires_grad propagation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## requires_grad propagation — quick refresher

Three-gate AND: toggle AND is_differentiable AND any-input-requires-grad. The any-input scan must include **kwargs Tensor values** too — `dim=` or `mask=tensor` style kwargs can be Tensors that should propagate grad.

**Worked exemplar.** `op(x_constant, mask=tracked_tensor)` — positional `x` has `requires_grad=False`, but kwarg `mask` has `requires_grad=True`. The output must have `requires_grad=True`.

### Exercise 2 — requires_grad propagation also scans kwargs Tensor values

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the three-gate AND (toggle, is_differentiable, any-input) where any-input scans BOTH positional args AND kwargs.values() for MiniTensors with requires_grad=True.
> Keywords: requires-grad, kwargs, three-gate, tensor-valued-kwarg
> ```

**KCs targeted:** `requires-grad-propagation`, `requires-grad-scan-includes-kwargs`

Implement `propagate_requires_grad_full(args, kwargs, is_differentiable, grad_tracking_enabled) -> bool`:

Return `True` IFF ALL of:
1. `grad_tracking_enabled` is `True`.
2. `is_differentiable` is `True`.
3. At least one MiniTensor in `args` OR in `kwargs.values()` has `requires_grad == True`.

Non-MiniTensor entries (ints, floats, lists, plain torch.Tensors) are filtered out — same as the positional case in ex1. The extension is that kwargs `.values()` go through the SAME filter AND OR-combine into the any-test.

**Why this matters.** Some forward ops take Tensor-valued kwargs (`F.conv2d(x, weight=W, bias=b)` style). If `x` has `requires_grad=False` but `W` has `requires_grad=True`, the output must propagate grad — the parameter is the one being differentiated. Ignoring kwargs would drop those gradient paths.

The test sweeps all 8 truth-table combinations of (toggle, is_diff, has-tracked-input) PLUS a focused set of pos-vs-kwarg any-input cases.

In [ ]:
def propagate_requires_grad_full(
    args: tuple,
    kwargs: dict,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    """Three-gate AND scanning both args and kwargs.values() for tracked MiniTensors."""
    raise NotImplementedError()


def _test_ex2():
    T1 = MiniTensor(t.tensor([1.0]), requires_grad=True)
    T0 = MiniTensor(t.tensor([1.0]), requires_grad=False)

    # --- happy path: all three gates True, kwarg-only tracked Tensor ---
    assert propagate_requires_grad_full(
        args=(T0,), kwargs={'mask': T1},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is True, 'kwarg-only tracked Tensor must propagate'

    # --- positional tracked, kwarg untracked ---
    assert propagate_requires_grad_full(
        args=(T1,), kwargs={'mask': T0},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is True

    # --- both tracked ---
    assert propagate_requires_grad_full(
        args=(T1,), kwargs={'mask': T1},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is True

    # --- NEITHER tracked (across BOTH pos and kwarg) ---
    assert propagate_requires_grad_full(
        args=(T0,), kwargs={'mask': T0},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is False

    # --- empty inputs ---
    assert propagate_requires_grad_full(
        args=(), kwargs={},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is False, 'no inputs at all → False'

    # --- non-MiniTensor kwargs values are skipped (not crashed on) ---
    assert propagate_requires_grad_full(
        args=(T0,), kwargs={'dim': 1, 'keepdim': False, 'mask': T1},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is True, 'int/bool kwargs must be skipped without crashing'

    # --- non-MiniTensor kwargs values, no tracked anywhere ---
    assert propagate_requires_grad_full(
        args=(T0,), kwargs={'dim': 1, 'keepdim': False},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is False, 'no Tensor anywhere → False'

    # --- gate 1 OFF: toggle is False ---
    assert propagate_requires_grad_full(
        args=(T1,), kwargs={'mask': T1},
        is_differentiable=True, grad_tracking_enabled=False,
    ) is False, 'toggle off vetoes everything'

    # --- gate 2 OFF: non-differentiable op ---
    assert propagate_requires_grad_full(
        args=(T1,), kwargs={'mask': T1},
        is_differentiable=False, grad_tracking_enabled=True,
    ) is False, 'is_differentiable=False vetoes (e.g. torch.equal)'

    # --- plain torch.Tensor in kwargs must NOT count as a parent ---
    # (the wrapper sees MiniTensor; raw torch.Tensor is non-input scaffold)
    raw = t.tensor([1.0])    # NOT a MiniTensor, has no requires_grad-tracked semantics here
    assert propagate_requires_grad_full(
        args=(T0,), kwargs={'x': raw},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is False, 'raw torch.Tensor in kwargs must be filtered out — not a MiniTensor'

    # --- multiple kwargs, only ONE tracked ---
    assert propagate_requires_grad_full(
        args=(), kwargs={'a': T0, 'b': T0, 'c': T1, 'd': T0},
        is_differentiable=True, grad_tracking_enabled=True,
    ) is True, 'any() over kwargs must find the one tracked Tensor'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def propagate_requires_grad_full(args, kwargs, is_differentiable, grad_tracking_enabled):
    any_tracked = any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    ) or any(
        isinstance(v, MiniTensor) and v.requires_grad for v in kwargs.values()
    )
    return grad_tracking_enabled and is_differentiable and any_tracked
```

**Why kwargs need the same scan.** Many real autograd ops take tensor-valued kwargs that ARE differentiable: `F.linear(x, weight=W, bias=b)`, attention's `mask` weight, etc. If `requires_grad` propagation only looked at positional args, calling `F.linear(x, weight=W)` with `x.requires_grad=False` but `W.requires_grad=True` would skip building a Recipe and lose the gradient path back to `W`.

**Filter on the input type, not the key.** A kwarg whose VALUE is a Tensor counts; a kwarg whose value is an int doesn't. The `isinstance(v, MiniTensor)` filter applied to `.values()` performs the same role as the positional `isinstance(a, MiniTensor)` filter — uniform semantics.

**Short-circuit ordering.** Putting the cheap gates first (`grad_tracking_enabled`, `is_differentiable`) lets Python's `and` short-circuit before the linear `any()` scan over inputs. Tiny win, but it's the natural ordering — global state first, then per-op flag, then per-input traversal.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()